# سبق 10 - پیداوار میں AI ایجنٹس

اس سبق میں آپ Microsoft Agent Framework (Python) استعمال کرنے والے AI ایجنٹس کے لیے **پیداواری نمونے** سیکھیں گے۔ ہم درج ذیل کا احاطہ کرتے ہیں:

- **مشاہدہ پذیری** — ایجنٹ کے تعاملات میں وقت ناپنے اور لاگنگ شامل کرنا
- **تشخیص** — جواب کے معیار کو اسکور کرنے کے لیے ایک جانچنے والے ایجنٹ کا استعمال
- **لاگت کا انتظام** — ٹوکن کی اصلاح اور ماڈل کے انتخاب کی حکمتِ عملیاں

منظرنامہ ایک **سفری ایجنٹ** ہے جو صارفین کو سفر کی منصوبہ بندی میں مدد دیتا ہے، جس کے اوپر نگرانی اور تشخیص کی تہہ موجود ہے۔


## ترتیب


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## پیداواری غور و فکر

AI ایجنٹس کو پروٹوٹائپس سے پیداوار تک منتقل کرنے کے لیے تین ستونوں پر محتاط توجہ درکار ہوتی ہے:

1. **قابلِ مشاہدہ ہونا** — آپ کو یہ دیکھنے کی ضرورت ہے کہ ایجنٹ کیا کر رہا ہے، اسے کتنا وقت لگتا ہے، اور وہ کون سے ٹولز کال کرتا ہے۔ بغیر ٹریسنگ اور لاگنگ کے، پیداوار میں مسائل کی خرابی تلاش کرنا تقریباً ناممکن ہے۔

2. **تسخیص** — خودکار معیار کی جانچیں یقینی بناتی ہیں کہ ایجنٹ کے جوابات وقت کے ساتھ درست، مکمل، اور مددگار رہیں۔ ایک تسخیصی ایجنٹ متعین کردہ معیار کے خلاف جوابات کو اسکور کر سکتا ہے۔

3. **لاگت کا انتظام** — ٹوکن کے استعمال کا براہِ راست اثر لاگت پر پڑتا ہے۔ پرامپٹ کی اصلاح، ماڈل کے انتخاب، اور کیشنگ جیسی حکمتِ عملیاں بغیر معیار کے سمجھوتے کے اخراجات کو قابو میں رکھنے میں مدد دیتی ہیں۔


## ایک قابل مشاہدہ ایجنٹ بنانا

ہم سفر کے اوزار متعین کرتے ہیں اور تاخیر کی نگرانی کے لیے ایجنٹ کال کو ٹائمنگ کے ساتھ لپیٹتے ہیں۔ پروڈکشن میں آپ OpenTelemetry یا اسی طرح کے کسی ٹریسنگ بیک اینڈ کے ساتھ انضمام کریں گے۔


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## جائزہ کے نمونے

پیداواری ماحول میں ایک عام طریقہ یہ ہے کہ دوسرے ایجنٹ کو **جانچنے والا** کے طور پر استعمال کیا جائے۔ جانچنے والا بنیادی ایجنٹ کے جواب کو پہلے سے طے شدہ معیار، جیسے مکملیت، درستگی، اور مفیدیت کے مطابق اسکور کرتا ہے۔

یہ ممکن بناتا ہے:
- جوابات صارفین تک پہنچنے سے پہلے خودکار معیار کے دروازے
- جب پرامپٹس یا ماڈلز تبدیل ہوں تو ریگریشن کا پتہ لگانا
- وقت کے ساتھ ایجنٹ کی کارکردگی کی مسلسل نگرانی


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## لاگت کے انتظام کی حکمت عملیاں

لاگت کو کنٹرول کرنا پیداواری AI ایجنٹس کے لیے انتہائی اہم ہے۔ یہاں اہم حکمت عملیاں دی گئی ہیں:

| حکمتِ عملی | وضاحت |
|---|---|
| **پرومپٹ کی اصلاح** | سسٹم ہدایات کو مختصر رکھیں۔ اضافی سیاق و سباق کو ہٹا دیں تاکہ ان پٹ ٹوکنز کم ہوں۔ |
| **ماڈل کا انتخاب** | سادہ کاموں جیسے درجہ بندی یا استخراج کے لیے چھوٹے، سستے ماڈلز استعمال کریں (مثلاً GPT-4o-mini)، اور پیچیدہ استدلال کے لیے بڑے ماڈلز کو محفوظ رکھیں۔ |
| **کیشنگ** | ٹول کے نتائج اور کثرت سے آنے والی استفسارات کو کیش کریں تاکہ غیر ضروری API کالز سے بچا جا سکے۔ |
| **ٹوکن بجٹس** | غیر متوقع طور پر طویل جوابات سے بچنے کے لیے `max_tokens` کی حدود مقرر کریں۔ |
| **بیچنگ** | جہاں ممکن ہو متعدد صارف کے استفسارات کو ایک واحد API کال میں گروپ کریں۔ |

عملی طور پر، ایک مرحلہ وار طریقۂ کار اچھا کام کرتا ہے: سیدھے سادے درخواستوں کو ایک تیز، سستا ماڈل تفویض کریں اور صرف پیچیدہ سوالات کو کسی زیادہ قابل ماڈل کی طرف منتقل کریں۔


## خلاصہ

اس سبق میں آپ نے یہ سیکھا کہ کیسے:

1. **مشاہدگی شامل کریں** ایجنٹ تعاملات میں ٹائمنگ اور لاگنگ کے ذریعے، جو ٹریسنگ اور مانیٹرنگ کے لیے بنیاد فراہم کرتی ہے۔
2. **ایجنٹ کے جوابات کا جائزہ لیں** خودکار طور پر ایک جائزہ لینے والے ایجنٹ کا استعمال کرتے ہوئے جو تکمیل، درستگی، اور مفیدیت کو اسکور کرتا ہے۔
3. **اخراجات کا انتظام کریں** پرامپٹ کی اصلاح، ماڈل کے انتخاب، کیشنگ، اور ٹوکن بجٹس کے ذریعے۔

یہ پروڈکشن پیٹرنز اس بات کو یقینی بنانے میں مدد دیتے ہیں کہ آپ کے AI ایجنٹس بڑے پیمانے پر قابلِ اعتماد، قابلِ پیمائش، اور لاگت کے لحاظ سے مؤثر ہوں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ڈسکلیمر:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہِ کرم نوٹ کریں کہ خودکار تراجم میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں معتبر ماخذ سمجھنی چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کے لیے ہم ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
